<a href="https://colab.research.google.com/github/Basharbalaa/chicago-food-inspections/blob/main/02_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/chicago-inspections/food_inspections.zip')

In [3]:
df["Inspection Date"] = pd.to_datetime(df["Inspection Date"], format = "%m/%d/%Y")

raw = df.copy()

In [4]:
df = df[df["Facility Type"] == "Restaurant"]
df.shape

(215955, 17)

In [5]:
import numpy as np
df["License #"] = df["License #"].replace(0, np.nan)
df["License #"].isnull().sum()

np.int64(211)

In [6]:
df["Results"].value_counts()

,count
Results,
Pass,109913
Fail,40231
Pass w/ Conditions,35401
Out of Business,15223
No Entry,12108
Not Ready,3049
Business Not Located,30


In [7]:
df.head(50)

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location
0,2646614,SEE THRU CHINESE KITCHEN,SEE THRU CHINESE KITCHEN,36468.0,Restaurant,Risk 1 (High),10939 S WESTERN AVE,CHICAGO,IL,60643.0,2026-11-10,Complaint Re-Inspection,Pass,37. FOOD PROPERLY LABELED; ORIGINAL CONTAINER ...,41.694221,-87.681086,"(41.69422064707933, -87.68108589547474)"
1,2646610,THE SIAM LOTUS THAI CUISINE LLC,THE SIAM LOTUS THAI CUISINE,3105008.0,Restaurant,Risk 1 (High),3445 W 51ST ST,CHICAGO,IL,60632.0,2026-11-10,License Re-Inspection,Pass,51. PLUMBING INSTALLED; PROPER BACKFLOW DEVICE...,41.800619,-87.710456,"(41.80061941881158, -87.71045584912062)"
2,2646574,"LA CIUDAD DE PIEDRADURA , INC.",LA CIUDAD DE PIEDRADURA,3100520.0,Restaurant,Risk 1 (High),4781 S ARCHER AVE,CHICAGO,IL,60632.0,2026-11-10,License,Fail,"1. PERSON IN CHARGE PRESENT, DEMONSTRATES KNOW...",41.806668,-87.716128,"(41.80666845625096, -87.71612754564387)"
3,2646594,FOOD EXPRESS,FOOD EXPRESS,3096350.0,Restaurant,Risk 1 (High),113 E 51ST ST,CHICAGO,IL,60615.0,2026-11-10,License Re-Inspection,Pass,NaN,41.801900,-87.622018,"(41.80190047384105, -87.62201755519536)"
6,2646608,KENTUCKY FRIED CHICKEN #5287,KFC,2523422.0,Restaurant,Risk 1 (High),1617 E 95TH ST,CHICAGO,IL,60617.0,2026-11-10,Canvass,Pass,"55. PHYSICAL FACILITIES INSTALLED, MAINTAINED ...",41.722241,-87.584380,"(41.722241316658945, -87.5843801363996)"
7,2646611,GREEN LEAF THAI CUISINE,GREEN LEAF THAI CUISINE,2601599.0,Restaurant,Risk 1 (High),4658 N WESTERN AVE,CHICAGO,IL,60625.0,2026-11-10,Canvass Re-Inspection,Fail,"38. INSECTS, RODENTS, & ANIMALS NOT PRESENT - ...",41.966501,-87.688987,"(41.96650114090258, -87.68898734901568)"
8,2646575,YAYA MAS GREEK KUZINA,YAYA MAS GREEK KUZINA,3083204.0,Restaurant,Risk 1 (High),2300 N MILWAUKEE AVE,CHICAGO,IL,60647.0,2026-11-10,License,Pass,47. FOOD & NON-FOOD CONTACT SURFACES CLEANABLE...,41.922843,-87.697487,"(41.922842576706046, -87.69748748180638)"
10,2646530,CALUMET FISHERIES INC,CALUMET FISHERIES INC,3543.0,Restaurant,Risk 1 (High),3259 E 95TH ST,CHICAGO,IL,60617.0,2026-11-09,Canvass,Pass,"55. PHYSICAL FACILITIES INSTALLED, MAINTAINED ...",41.722733,-87.544531,"(41.72273269460603, -87.5445308299031)"
11,2646526,REILLY'S DAUGHTER IRISH PUB,REILLY'S DAUGHTER IRISH PUB,2584039.0,Restaurant,Risk 1 (High),5700 S CICERO AVE,CHICAGO,IL,60638.0,2026-11-09,Canvass,Pass,"55. PHYSICAL FACILITIES INSTALLED, MAINTAINED ...",41.789329,-87.741646,"(41.789329323265385, -87.74164564419637)"
12,2646521,CIELO'S CANTINA,CIELO'S CANTINA,2895471.0,Restaurant,Risk 1 (High),5700 S CICERO AVE,CHICAGO,IL,60638.0,2026-11-09,Canvass,Pass,49. NON-FOOD/FOOD CONTACT SURFACES CLEAN - Com...,41.789329,-87.741646,"(41.789329323265385, -87.74164564419637)"


In [8]:
real = ["Pass", "Pass w/ Conditions", "Fail"]
other = df[~df["Results"].isin(real)]
other["Inspection Type"].value_counts().head(8)

,count
Inspection Type,
Canvass,19800
Non-Inspection,5117
License,3323
Complaint,1464
Canvass Re-Inspection,293
Complaint Re-Inspection,140
Short Form Complaint,128
License Re-Inspection,49


In [9]:
df = df[df["Results"].isin(real)].copy()
df["Failed"] = (df["Results"] == "Fail").astype(int)
df["Failed"].value_counts()

,count
Failed,
0,145314
1,40231


In [10]:
df.shape

(185545, 18)

In [11]:
df.groupby("Inspection Type")["Failed"].agg(["count", "mean"]).sort_values("count", ascending=False).head(10)

,count,mean
Inspection Type,,
Canvass,91552,0.223010
Canvass Re-Inspection,24100,0.071494
Complaint,22286,0.341111
License,20760,0.342582
Complaint Re-Inspection,9497,0.110982
License Re-Inspection,7105,0.085011
Short Form Complaint,6969,0.131583
Suspected Food Poisoning,870,0.266667
Tag Removal,482,0.043568


In [12]:
df["Year"] = df["Inspection Date"].dt.year
df.groupby("Year")["Failed"].mean().round(3)

,Failed
Year,
2010,0.240
2011,0.230
2012,0.213
2013,0.185
2014,0.190
2015,0.201
2016,0.210
2017,0.220
2018,0.210


In [13]:
df["Is_Reinspection"] = df["Inspection Type"].str.contains("Re-Inspection", na=False)
df["Is_Reinspection"].value_counts()

,count
Is_Reinspection,
False,144843
True,40702


In [14]:
df.groupby("Year")["Is_Reinspection"].mean().round(3)

,Is_Reinspection
Year,
2010,0.189
2011,0.195
2012,0.204
2013,0.189
2014,0.195
2015,0.213
2016,0.226
2017,0.232
2018,0.222


In [15]:
df.to_csv("/content/drive/MyDrive/chicago-inspections/clean_restaurants.csv.gz", index=False)

In [16]:
check = pd.read_csv("/content/drive/MyDrive/chicago-inspections/clean_restaurants.csv.gz", parse_dates=["Inspection Date"])
check.shape

(185545, 20)